<a href="https://colab.research.google.com/github/eeeewyz/Audio-course/blob/main/Application/15_STST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
from transformers import pipeline
device = "cuda:0" if torch.cuda.is_available() else "cpu"

#一：用TTS进行翻译
#加载asr模型
pipe = pipeline(
    "automatic-speech-recognition", model="openai/whisper-base", device=device
)

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

In [55]:
#加载一个意大利语的数据集，测试翻译情况，加载这个数据集："qmeeus/voxpopuli"
from datasets import load_dataset

dataset = load_dataset(
    "facebook/voxpopuli",
    "it",
    split="validation",
    streaming=True
)

it = iter(dataset)

sample1 = next(it)   # 第1条sample
sample2 = next(it)   # 第2条sample

In [56]:
#播放音频
from IPython.display import Audio

Audio(sample1["audio"]["array"], rate=sample1["audio"]["sampling_rate"])

In [57]:
Audio(sample2["audio"]["array"], rate=sample1["audio"]["sampling_rate"])

In [6]:
#任务一：若用asr转录
result = pipe(
    {
        "array": sample1["audio"]["array"],
        "sampling_rate": sample1["audio"]["sampling_rate"]
    },
    generate_kwargs={
        "language": "italian",
        "task": "transcribe"
    }
)

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [7]:
print(result["text"])

 psicologico e sociale, penso che sia un passo avanti importante nella costruzione di uno spazio juridico, di libertà di circolazione e di protezione dei diritti.


In [8]:
#任务二若用翻译功能，只要把pipe的任务类型改下
def translate(audio):
    outputs = pipe(audio, max_new_tokens=256, generate_kwargs={"task": "translate"})
    return outputs["text"]

In [9]:
translate(sample1["audio"])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


' psychological and social. I think that it is an important step in the construction of a juridical space of freedom, circulation and protection of rights.'

In [10]:
#二：加载tts模型
#加载speechT5模型
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

[transformers] SpeechT5ForTextToSpeech LOAD REPORT from: microsoft/speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] You are using a model of type `hifigan` to instantiate a model of type `speecht5_hifigan`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

In [11]:
#设备转移到GPU上
model.to(device)
vocoder.to(device)

SpeechT5HifiGan(
  (conv_pre): Conv1d(80, 512, kernel_size=(7,), stride=(1,), padding=(3,))
  (upsampler): ModuleList(
    (0): ConvTranspose1d(512, 256, kernel_size=(8,), stride=(4,), padding=(2,))
    (1): ConvTranspose1d(256, 128, kernel_size=(8,), stride=(4,), padding=(2,))
    (2): ConvTranspose1d(128, 64, kernel_size=(8,), stride=(4,), padding=(2,))
    (3): ConvTranspose1d(64, 32, kernel_size=(8,), stride=(4,), padding=(2,))
  )
  (resblocks): ModuleList(
    (0): HifiGanResidualBlock(
      (convs1): ModuleList(
        (0): Conv1d(256, 256, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): Conv1d(256, 256, kernel_size=(3,), stride=(1,), padding=(3,), dilation=(3,))
        (2): Conv1d(256, 256, kernel_size=(3,), stride=(1,), padding=(5,), dilation=(5,))
      )
      (convs2): ModuleList(
        (0-2): 3 x Conv1d(256, 256, kernel_size=(3,), stride=(1,), padding=(1,))
      )
    )
    (1): HifiGanResidualBlock(
      (convs1): ModuleList(
        (0): Conv1d(256, 256,

In [12]:
#看下模型基本结构
print(model)

SpeechT5ForTextToSpeech(
  (speecht5): SpeechT5Model(
    (encoder): SpeechT5EncoderWithTextPrenet(
      (prenet): SpeechT5TextEncoderPrenet(
        (embed_tokens): Embedding(81, 768, padding_idx=1)
        (encode_positions): SpeechT5ScaledPositionalEncoding(
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (wrapped_encoder): SpeechT5Encoder(
        (layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (layers): ModuleList(
          (0-11): 12 x SpeechT5EncoderLayer(
            (attention): SpeechT5Attention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (dropout): Dropout(p=0.1, inplace=False)
     

In [15]:
#加入speech embedding
!pip install -U "datasets<4" #先用旧版，目前新版不支持


embeddings_dataset = load_dataset(
    "Matthijs/cmu-arctic-xvectors",
    split="validation"
)


In [19]:
#看看这个speech embedding共有多少
len(embeddings_dataset)

7931

In [20]:
#用7306号embedding
speaker_embeddings = torch.tensor(
    embeddings_dataset[7306]["xvector"]
).unsqueeze(0)

In [30]:
def synthesise(text):
    # 1. 将输入文本转换成模型需要的 input_ids
    inputs = processor(text=text, return_tensors="pt")

    # 2. 输入文本 token IDs + speaker embedding
    #    SpeechT5 生成 Mel 特征，并通过 vocoder 转成 waveform
    speech = model.generate_speech(
        inputs["input_ids"].to(device),
        speaker_embeddings.to(device),
        vocoder=vocoder
    )
    # 3. 将生成的语音 tensor 从 GPU 移回 CPU
    return speech.cpu()


In [31]:
small_text="Hey there! This is a test!"

In [35]:
speech=synthesise(small_text)

In [37]:
Audio(speech, rate=16000)


In [39]:
# 模型输出的 waveform 通常是 float（约 [-1,1]），适合神经网络计算。
# 为了让音频播放器/文件格式更方便识别和保存，通常把它转换成标准的 int16 PCM（-32768~32767）。

# PyTorch float waveform
#         ↓ .numpy()
# NumPy float waveform
#         ↓ × 32767
# 放大到 PCM 数值范围
#         ↓ astype(np.int16)
# 16-bit PCM waveform

In [40]:
#三：开始STST，把之前两部分结合起来
import numpy as np

# 设置输出音频的数据类型为 int16
target_dtype = np.int16

# 获取 int16 的最大值：32767
max_range = np.iinfo(target_dtype).max

In [48]:
def speech_to_speech_translation(audio):

    # Hugging Face dataset audio格式
    audio_input = {
        "raw": audio["array"],
        "sampling_rate": audio["sampling_rate"]
    }

    # 1. 输入语音 → ASR + 翻译
    translated_text = translate(audio_input)

    # 2. 翻译文本 → TTS
    synthesised_speech = synthesise(translated_text)

    # 3. float waveform → int16 PCM
    synthesised_speech = (
        synthesised_speech.numpy() * max_range
    ).astype(np.int16)

    return 16000, synthesised_speech

In [58]:
#看下我最早的sample1和字段
print(sample1["audio"])
print(sample1["audio"].keys())

{'path': '20130522-0900-PLENARY-9-it_20130522-13:08:49_3.wav', 'array': array([-0.00064087,  0.02093506, -0.02679443, ..., -0.00280762,
       -0.00296021, -0.00247192]), 'sampling_rate': 16000}
dict_keys(['path', 'array', 'sampling_rate'])


In [59]:
#加载第一条audio，的意大利语，完成stst，翻译为英语
sampling_rate, synthesised_speech = speech_to_speech_translation(sample1["audio"])

Audio(synthesised_speech, rate=sampling_rate)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [60]:
#加载第二条audio，的意大利语，完成stst，翻译为英语
sampling_rate, synthesised_speech = speech_to_speech_translation(sample2["audio"])

Audio(synthesised_speech, rate=sampling_rate)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
